In [25]:
os.makedirs("results", exist_ok=True)

summary = pd.DataFrame([

{

"RQ":"RQ1",

"Metric":"Syntax Accuracy",

"MetricValue":syntax_accuracy,

"PValue":binom_p,

"EffectSize":np.nan,

"N":N,

"Decision":syntax_decision

},

{

"RQ":"RQ2",

"Metric":"BLEU",

"MetricValue":valid_bleu.mean(),

"PValue":wilcox_p,

"EffectSize":cohens_d,

"N":len(valid_bleu),

"Decision":bleu_decision

}

])

summary.to_csv(
    "results/summary.csv",
    index=False
)

print(summary)

    RQ           Metric  MetricValue        PValue  EffectSize   N   Decision
0  RQ1  Syntax Accuracy     0.000000  1.427248e-55         NaN  50  Reject H0
1  RQ2             BLEU     0.033846  2.580132e-08  -21.830897  39  Reject H0


In [24]:
alpha = 0.05

syntax_decision = (
    "Reject H0"
    if binom_p < alpha
    else "Fail to Reject H0"
)

bleu_decision = (
    "Reject H0"
    if wilcox_p < alpha
    else "Fail to Reject H0"
)

In [23]:
d = abs(cohens_d)

if d < 0.2:

    effect = "Small"

elif d < 0.5:

    effect = "Medium"

else:

    effect = "Large"

print(effect)

Large


In [28]:
import numpy as np

def cliffs_delta(x, y):
    gt = 0
    lt = 0

    for xi in x:
        for yi in y:
            if xi > yi:
                gt += 1
            elif xi < yi:
                lt += 1

    delta = (gt - lt) / (len(x) * len(y))

    return delta

TARGET = 0.7670

target = np.full(len(valid_bleu), TARGET)

delta = cliffs_delta(valid_bleu.values, target)

print("Cliff's Delta =", delta)

abs_delta = abs(delta)

if abs_delta < 0.147:
    effect = "Negligible"
elif abs_delta < 0.33:
    effect = "Small"
elif abs_delta < 0.474:
    effect = "Medium"
else:
    effect = "Large"

print("Effect Size =", effect)

Cliff's Delta = -1.0
Effect Size = Large


In [21]:
TARGET = 0.7670

difference = valid_bleu - TARGET

wilcox = wilcoxon(
    difference,
    alternative="less"
)

wilcox_p = wilcox.pvalue

print(wilcox)

WilcoxonResult(statistic=np.float64(0.0), pvalue=np.float64(2.5801318810448244e-08))


In [20]:
success = syntax_mask.sum()

result = binomtest(
    success,
    N,
    p=0.92,
    alternative="less"
)

print(result)

binom_p = result.pvalue

BinomTestResult(k=0.0, n=50.0, alternative='less', statistic=0.0, pvalue=1.4272476927059242e-55)


In [19]:
valid_bleu = llm["BLEU"].dropna()

print("Average BLEU :", valid_bleu.mean())

print("Median BLEU :", valid_bleu.median())

print("Min BLEU :", valid_bleu.min())

print("Max BLEU :", valid_bleu.max())

Average BLEU : 0.03384597985990362
Median BLEU : 0.030340187271410496
Min BLEU : 0.0
Max BLEU : 0.20389696163541748


In [18]:
smooth = SmoothingFunction().method1

bleu_scores = []

for output, gt in zip(
        llm["generated_gherkin"],
        truth["expected_scenario"]):

    if pd.isna(output):

        bleu_scores.append(np.nan)

        continue

    reference = [str(gt).split()]

    candidate = str(output).split()

    score = sentence_bleu(
        reference,
        candidate,
        smoothing_function=smooth
    )

    bleu_scores.append(score)

llm["BLEU"] = bleu_scores

In [17]:
N = len(llm)

valid_mask = llm["record_status"] == "VALID"

syntax_mask = llm["syntax_pass"] == True

syntax_accuracy = syntax_mask.mean()

invalid_rate = (~valid_mask).mean()

print("Total:", N)
print("Syntax Accuracy:", syntax_accuracy)
print("Invalid Rate:", invalid_rate)

Total: 50
Syntax Accuracy: 0.0
Invalid Rate: 0.22


In [ ]:
llm = pd.read_csv("full_llm_output.csv")

truth = pd.read_csv(r"D:\long\swt\lab4\nhom_7\data\raw\full_ground_truth.csv")


In [2]:
import os
import numpy as np
import pandas as pd

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

from scipy.stats import binomtest
from scipy.stats import wilcoxon